In [61]:
import pyautogui as pyg
pyg.sleep(2)
basex=337
basey=378
gap=40

In [62]:
def gem_position(r, c):
    return (basex + c * gap, basey + r * gap)

In [63]:
print(gem_position(0,0))
print(gem_position(0,1))
print(gem_position(0,2))

(337, 378)
(377, 378)
(417, 378)


In [64]:
im=pyg.screenshot()
for i in range(8):
    x ,y = gem_position(0,i)
    pyg.click(x,y,duration=1)
    print(get_gem_color(0,i,im))

(70, 245, 115)
(18, 137, 251)
(253, 250, 37)
(253, 250, 37)
(251, 251, 251)
(253, 28, 57)
(70, 245, 115)
(251, 251, 251)


In [65]:
def get_gem_color(r, c, im):
    x, y = gem_position(r, c)
    return im.getpixel((x, y))

In [66]:
def can_move_left(r, c, im):
    """判斷第 r 列、第 c 欄的寶石往左交換後是否會形成三連。"""
    if not (0 <= r < 8 and 1 <= c < 8):
        return False

    p1 = (r, c)
    p2 = (r, c - 1)
    color1 = get_gem_color(r, c, im)
    color2 = get_gem_color(r, c - 1, im)

    def color_after_swap(row, col):
        if (row, col) == p1:
            return color2
        if (row, col) == p2:
            return color1
        return get_gem_color(row, col, im)

    for row, col in (p1, p2):
        color = color_after_swap(row, col)
        for dr, dc in ((0, 1), (1, 0)):
            count = 1
            for direction in (-1, 1):
                nr, nc = row + dr * direction, col + dc * direction
                while 0 <= nr < 8 and 0 <= nc < 8 and color_after_swap(nr, nc) == color:
                    count += 1
                    nr += dr * direction
                    nc += dc * direction
            if count >= 3:
                return True

    return False

In [67]:
im=pyg.screenshot()
print(can_move_left(0,1,im))

True


In [ ]:
def _can_move_in_direction(r, c, im, dr, dc):
    """判斷寶石與指定方向的相鄰寶石交換後是否會形成三連。"""
    nr, nc = r + dr, c + dc
    if not (0 <= r < 8 and 0 <= c < 8 and 0 <= nr < 8 and 0 <= nc < 8):
        return False

    p1, p2 = (r, c), (nr, nc)
    color1 = get_gem_color(r, c, im)
    color2 = get_gem_color(nr, nc, im)

    def color_after_swap(row, col):
        if (row, col) == p1:
            return color2
        if (row, col) == p2:
            return color1
        return get_gem_color(row, col, im)

    for row, col in (p1, p2):
        color = color_after_swap(row, col)
        for match_dr, match_dc in ((0, 1), (1, 0)):
            count = 1
            for direction in (-1, 1):
                rr = row + match_dr * direction
                cc = col + match_dc * direction
                while (
                    0 <= rr < 8
                    and 0 <= cc < 8
                    and color_after_swap(rr, cc) == color
                ):
                    count += 1
                    rr += match_dr * direction
                    cc += match_dc * direction
            if count >= 3:
                return True

    return False


def can_move_right(r, c, im):
    return _can_move_in_direction(r, c, im, 0, 1)


def can_move_up(r, c, im):
    return _can_move_in_direction(r, c, im, -1, 0)


def can_move_down(r, c, im):
    return _can_move_in_direction(r, c, im, 1, 0)